# Lesson 3: Text summarization with Titan Text Premier

In this notebook, you ingest a small string of text directly into the Amazon Bedrock API (using the Titan Text model) and instruct it to summarize the input text. You can apply this approach to summarize call transcripts, meeting transcripts, books, articles, blog posts, and other relevant content when the input text length is within the context size limits of the model.

## Task 3.1: Environment setup

In this task, you set up your environment.

In [0]:
import json
import os
import sys
import warnings

warnings.filterwarnings('ignore')
module_path = ".."
sys.path.append(os.path.abspath(module_path))

The `boto3.client('bedrock-runtime')` method creates a client for interacting with Amazon Bedrock's model inference APIs. It accepts the following:

- `service_name`: The AWS service to connect to, in this case, `bedrock-runtime`.
- `region_name`: The AWS Region where the service is available. It defaults to the value of the `AWS_DEFAULT_REGION` environment variable, or `None` if the variable is not set, allowing Boto3 to resolve the region from its default configuration.

Refer to the [Boto3 documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/core/session.html?utm_source=chatgpt.com#boto3.session.Session.client) for more information.

In [0]:
import boto3
import botocore

bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name=os.environ.get("AWS_DEFAULT_REGION", None)
)

## Task 3.2: Project Setup

In this section, we'll proceed with the initial configuration of the project.

The cell below contains a short passage with fewer tokens than the foundation model's maximum context length. For this lab, the sample input is a paragraph reflecting on the song *Upside Down* by Jack Johnson.

The prompt begins with the instruction: `Please provide a summary of the following text.`

In [0]:
prompt_data = """
Please provide a summary using of the following text:

Jack Johnson’s "Upside Down" is a gentle manifesto on curiosity, human perception, and the power \
of changing one’s perspective. At its core, the song suggests that the world is not a fixed structure \
to be endured, but an evolving landscape to be explored. A central theme of the track is the courage \
to question the status quo. The metaphor of turning the world upside down represents a willingness \
to challenge default assumptions, encouraging people to adopt a childlike mind that continuously asks \
questions and refuses to accept rigid routines simply because that is how things have always been. \
Furthermore, the song emphasizes embracing the unknown over the fear of failure. It frames exploration \
as a source of joy rather than a risky gamble, illustrating that stepping outside predictable boundaries \
and making mistakes through trial and error are essential parts of learning and self-discovery. \
Ultimately, the song reminds us that perception shapes reality. By shifting from a passive or defensive \
stance to an active and curious one, ordinary moments can reveal hidden beauty and possibilities. Living \
upside down means practicing deliberate open-mindedness, finding joy in learning new things, and \
approaching daily challenges with patience, optimism, and lightheartedness.
"""

The code below defines the model ID and constructs the JSON request body, including the input prompt and text generation parameters. This configuration will be used to invoke Amazon Titan Text Premier through Amazon Bedrock, supporting both standard and streaming output modes.

In [0]:
modelId = "amazon.nova-micro-v1:0"

accept = "application/json"
contentType = "application/json"
inferenceConfig = {
        "maxTokens": 2048,
        "stopSequences": [],
        "temperature": 0,
        "topP": 0.9
    }
body = json.dumps({
    "messages":  [{
        "role": "user",
        "content":  [{ "text": prompt_data }]
    }], 
    "inferenceConfig": inferenceConfig
}) 

request_body = {
    "modelId": modelId,
    "accept": accept,
    "contentType": contentType,
    "body": body
}

## Task 3.3: Invoke foundation model via Boto3

In this task, you send an API request to Amazon Bedrock specifying the request parameters: `modelId`, `accept`, and `contentType`. Following the provided prompt, the foundation model in Amazon Bedrock then summarizes the input text.

### Complete Output Generation

By default, the Amazon Bedrock service generates the entire summary for a given prompt in a single output. This can be slow if the model output contains many tokens. 

In [0]:
try:
    response = bedrock_client.invoke_model(**request_body)
    response_body = json.loads(response.get('body').read())

    outputText = response_body['output']['message']['content'][0]['text']
except botocore.exceptions.ClientError as err:
    print(f"\x1b[41m{err.response['Error']['Message']}\
        \nTo troubeshoot this issue please refer to the following resources.\
            \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
    ")
except Exception as err:
        raise err

print("Assistant:", outputText)

Assistant: Jack Johnson’s "Upside Down" is a song that advocates for curiosity, questioning, and changing one’s perspective. It promotes the idea that the world is an ever-changing landscape rather than a fixed structure. The song encourages listeners to challenge the status quo and adopt a mindset of continuous questioning, akin to a child’s inquisitive nature. It highlights the importance of embracing the unknown and viewing exploration as a joyful process rather than a fearful one. By stepping outside of predictable routines and making mistakes through trial and error, individuals can learn and discover new possibilities. The song ultimately suggests that our perception shapes our reality, and by adopting an active, curious, and open-minded approach to life, we can uncover hidden beauty and opportunities in everyday moments. Living "upside down" means embracing open-mindedness, finding joy in learning, and approaching challenges with patience, optimism, and a lighthearted attitude.


### Streaming Output Generation

Next, you explore how to use Amazon Bedrock's invoke_model_with_response_stream API to stream model outputs so users can consume outputs as they are generated. Rather than generating the full output at once, this API returns a ResponseStream that sends smaller output chunks from the model as they are produced. You can display these streaming outputs in a continuous, consumable view.

In [0]:
response = bedrock_client.invoke_model_with_response_stream(**request_body)
stream = response.get('body')

print('Assistant:', end="")
if stream:
    for event in stream:
        chunk = json.loads(event.get('chunk').get('bytes'))

        if delta_event := chunk.get('contentBlockDelta'):
            message = delta_event['delta'].get('text')
            print(message, end="")


Assistant:Jack Johnson’

s "Upside Down" advocates for curiosity, questioning, and a flexible perspective on life. The song promotes the idea that the world is not static but dynamic and worth exploring. It encourages listeners to challenge conventional wisdom and adopt a questioning, childlike mindset

 that resists rigid routines. The metaphor of turning the world upside down symbolizes a willingness to confront and rethink default assumptions. The track highlights the importance of embracing the unknown and viewing exploration as a joyful journey rather than a fearful

 endeavor. It suggests that making mistakes through trial and error is a crucial part of learning and self-discovery. Ultimately, the song emphasizes that our perception shapes our reality, and by adopting an active, curious, and open-minded approach, we can uncover hidden beauty and possibilities in everyday life. Living "upside down" means approaching life with patience, optimism, and a lighthearted spirit.

You have now experimented with using the boto3 SDK to access the Amazon Bedrock API. This SDK provides basic programmatic access to Bedrock capabilities. By leveraging this API, you were able to implement two use cases: 1) Generating an entire text summary of AWS news content at once, and 2) Streaming the summary output in chunks for incremental processing.

### Try it yourself
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

I hope you enjoyed it!

Visit my [GitHub profile](https://github.com/rafaself).